In [ ]:
#!/usr/bin/env python3
"""
Interactive Bayesian Optimization Color Matching Simulation
- Allows user to choose a target color
- Simulates BO iteratively, printing CMYK values at each step
- Visualizes progress toward the target color
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
import random
import time

# Color conversion functions
def RGB_to_CMYK(rgb):
    """Convert RGB in [0,1] to CMYK in [0,1]"""
    r, g, b = rgb
    # Black (K) calculation
    k = 1 - max(r, g, b)
    # Handle pure black case
    if k == 1:
        return np.array([0, 0, 0, 1])
    # CMY calculation
    c = (1 - r - k) / (1 - k)
    m = (1 - g - k) / (1 - k)
    y = (1 - b - k) / (1 - k)
    return np.array([c, m, y, k])

def CMYK_to_RGB(cmyk):
    """Convert CMYK in [0,1] to RGB in [0,1]"""
    c, m, y, k = cmyk
    r = (1 - c) * (1 - k)
    g = (1 - m) * (1 - k)
    b = (1 - y) * (1 - k)
    return np.array([r, g, b])

def get_target_color():
    """Let user choose a target color"""
    print("\n=== Target Color Selection ===")
    print("Enter RGB values (0-255) for your target color")
    
    # Option to use a predefined color
    preset_option = input("Use a preset color? (y/n): ").strip().lower()
    if preset_option == 'y':
        presets = {
            '1': [255, 0, 0],    # Red
            '2': [0, 255, 0],    # Green
            '3': [0, 0, 255],    # Blue
            '4': [255, 255, 0],  # Yellow
            '5': [0, 255, 255],  # Cyan
            '6': [255, 0, 255],  # Magenta
            '7': [128, 0, 128],  # Purple
            '8': [165, 42, 42],  # Brown
            '9': [255, 165, 0],  # Orange
            '10': [75, 46, 131], # Deep Purple
        }
        
        print("Available preset colors:")
        for key, color in presets.items():
            print(f"{key}: RGB{tuple(color)}")
            
        choice = input("Enter your choice (1-10): ").strip()
        if choice in presets:
            target_rgb = presets[choice]
        else:
            print("Invalid choice, using default (red)")
            target_rgb = [255, 0, 0]
    else:
        # Manual RGB entry
        try:
            r = int(input("Red (0-255): ").strip())
            g = int(input("Green (0-255): ").strip())
            b = int(input("Blue (0-255): ").strip())
            target_rgb = [max(0, min(255, r)), max(0, min(255, g)), max(0, min(255, b))]
        except ValueError:
            print("Invalid input, using default (red)")
            target_rgb = [255, 0, 0]
    
    # Normalize to [0,1] range
    target_rgb_norm = np.array(target_rgb) / 255.0
    target_cmyk = RGB_to_CMYK(target_rgb_norm)
    
    print(f"\nTarget color: RGB{target_rgb} → CMYK[{', '.join([f'{v:.3f}' for v in target_cmyk])}]")
    
    # Display the target color
    plt.figure(figsize=(2, 2))
    plt.imshow([[target_rgb_norm]], extent=[0, 1, 0, 1])
    plt.axis("off")
    plt.title("Target Color")
    plt.show()
    
    return target_rgb, target_rgb_norm, target_cmyk

def generate_initial_samples(n_samples=1, n_features=4):
    """Generate initial CMYK sample(s) in [0,1]"""
    # Use Latin Hypercube Sampling for better coverage
    samples = np.random.rand(n_samples, n_features)
    
    # Normalize so they sum to reasonable values
    for i in range(n_samples):
        # Scale down if sum > 2.0 (arbitrary threshold to prevent all-black)
        if np.sum(samples[i]) > 2.0:
            samples[i] = samples[i] * (2.0 / np.sum(samples[i]))
    
    return samples

def expected_improvement(X, X_sample, y_sample, model, hyperparameter=0.01):
    """
    Expected Improvement acquisition function
    
    Args:
        X: Points to evaluate EI at
        X_sample: Sample points observed so far
        y_sample: Values observed at sample points
        model: Fitted GP model
        hyperparameter: Exploration vs exploitation parameter
    """
    mu, sigma = model.predict(X, return_std=True)
    
    # Find current best
    optimal_value = np.max(y_sample)
    
    # Calculate improvement
    with np.errstate(divide="ignore"):
        imp = mu - optimal_value - hyperparameter
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    
    return ei.flatten()

def propose_location(acquisition, X_sample, y_sample, bounds, gp, n_restarts=10):
    """Propose the next sampling point by optimizing the acquisition function"""
    dim = X_sample.shape[1]
    
    def min_obj(x):
        return -acquisition(x.reshape(1, -1), X_sample, y_sample, gp)
    
    # Starting points for optimization
    x0_list = np.random.uniform(0, 1, size=(n_restarts, dim))
    
    # Add the best point so far as a starting point
    best_idx = np.argmax(y_sample)
    x0_list = np.vstack((x0_list, X_sample[best_idx]))
    
    best_x, best_val = None, float("inf")
    
    # Run optimization from each starting point
    for x0 in x0_list:
        res = minimize(min_obj, x0, bounds=bounds, method="L-BFGS-B")
        if res.fun < best_val:
            best_val = res.fun
            best_x = res.x
            
    return best_x

def visualize_all_iterations(X_samples_rgb, distances, target_rgb_norm):
    """Visualize all color iterations and convergence plot"""
    n_samples = len(X_samples_rgb)
    colors_per_row = min(10, n_samples)
    n_rows = (n_samples + colors_per_row - 1) // colors_per_row
    
    plt.figure(figsize=(12, max(2, n_rows * 1.2)))
    for idx, rgb in enumerate(X_samples_rgb):
        row = idx // colors_per_row
        col = idx % colors_per_row
        
        plt.subplot(n_rows, colors_per_row, idx + 1)
        plt.imshow([[rgb]], extent=[0, 1, 0, 1])
        plt.axis("off")
        plt.title(f"Iter {idx + 1}", fontsize=8)
        
    plt.tight_layout()
    plt.show()
    
    # Plot convergence
    plt.figure(figsize=(8, 6))
    plt.plot(range(1, len(distances) + 1), distances, marker='o', linestyle='-', color='b')
    plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    plt.title('Convergence of Bayesian Optimization')
    plt.xlabel('Iteration')
    plt.ylabel('Distance to Target Color')
    plt.grid(True)
    plt.show()

def display_cmyk_volumes(cmyk, max_volume=270, title="CMYK Volumes"):
    """Display CMYK volumes in microliters"""
    volumes = [round(val * max_volume, 2) for val in cmyk]
    
    print(f"\n=== {title} ===")
    print(f"Cyan: {volumes[0]}µL")
    print(f"Magenta: {volumes[1]}µL")
    print(f"Yellow: {volumes[2]}µL")
    print(f"Black: {volumes[3]}µL")
    print(f"Total: {sum(volumes):.2f}µL")
    
    return volumes

def compare_colors(cmyk1, cmyk2, labels=["Color 1", "Color 2"]):
    """Compare two CMYK colors side by side"""
    rgb1 = CMYK_to_RGB(cmyk1)
    rgb2 = CMYK_to_RGB(cmyk2)
    
    plt.figure(figsize=(4, 2))
    plt.subplot(1, 2, 1)
    plt.imshow([[rgb1]], extent=[0, 1, 0, 1])
    plt.axis("off")
    plt.title(labels[0])
    
    plt.subplot(1, 2, 2)
    plt.imshow([[rgb2]], extent=[0, 1, 0, 1])
    plt.axis("off")
    plt.title(labels[1])
    
    plt.tight_layout()
    plt.show()

def run_interactive_bo():
    """Run interactive Bayesian Optimization for color matching"""
    # System parameters
    max_well_volume = 270  # Maximum volume in µL
    max_iterations = 20
    hyperparameter = 0.01  # Exploration vs exploitation tradeoff
    threshold = 0.02       # Target distance threshold to stop early
    
    # Get target color from user
    target_rgb, target_rgb_norm, target_cmyk = get_target_color()
    
    # Setup BO parameters
    n_features = 4  # CMYK components
    bounds = [(0, 1)] * n_features  # CMYK values normalized between 0 and 1
    
    # Generate initial sample(s)
    n_initial = 1
    X_initial = generate_initial_samples(n_samples=n_initial)
    
    # Calculate initial distance to target
    y_initial = [-np.linalg.norm(x - target_cmyk) for x in X_initial]
    
    # Initialize tracking variables
    X = X_initial.copy()  # All CMYK samples
    y = y_initial.copy()  # Corresponding negative distances
    
    X_samples = X.copy()  # Just to track colors at each step
    distances = [-yi for yi in y]  # Positive distances for plotting
    
    # Display initial sample
    print("\n=== Initial CMYK Sample ===")
    initial_cmyk = X[0]
    initial_rgb = CMYK_to_RGB(initial_cmyk)
    print(f"Initial CMYK: [{', '.join([f'{v:.3f}' for v in initial_cmyk])}]")
    print(f"Initial distance to target: {distances[0]:.4f}")
    
    display_cmyk_volumes(initial_cmyk, max_well_volume, "Initial Volumes")
    
    compare_colors(initial_cmyk, target_cmyk, ["Initial", "Target"])
    
    # Initialize GP model
    kernel = C(1.0, (1e-3, 1e3)) * RBF([0.1, 0.1, 0.1, 0.1], (1e-2, 1e2))
    gp = GaussianProcessRegressor(
        kernel=kernel, 
        n_restarts_optimizer=10, 
        alpha=1e-6,
        normalize_y=True,
        random_state=42
    )
    
    # Interactive BO loop
    iteration = 0
    best_distance = min(distances)
    continue_bo = True
    
    while continue_bo and iteration < max_iterations:
        # Fit GP model to all data so far
        gp.fit(X, y)
        
        # Propose next sampling point using EI
        next_cmyk = propose_location(expected_improvement, X, y, bounds, gp)
        
        # Calculate distance for this new point
        next_distance = np.linalg.norm(next_cmyk - target_cmyk)
        
        # Convert to RGB for visualization
        next_rgb = CMYK_to_RGB(next_cmyk)
        
        # Display suggestion
        iteration += 1
        print(f"\n=== Iteration {iteration} ===")
        print(f"Suggested CMYK: [{', '.join([f'{v:.3f}' for v in next_cmyk])}]")
        
        # Display volumes
        display_cmyk_volumes(next_cmyk, max_well_volume, f"Iteration {iteration} Volumes")
        
        # Show color comparison
        compare_colors(next_cmyk, target_cmyk, [f"Iteration {iteration}", "Target"])
        
        print(f"Distance to target: {next_distance:.4f}")
        
        # Check if we've improved
        if next_distance < best_distance:
            improvement = best_distance - next_distance
            print(f"Improved by {improvement:.4f}!")
            best_distance = next_distance
        else:
            print("No improvement in this iteration.")
        
        # Add new point to our dataset
        X = np.vstack((X, next_cmyk))
        y = np.append(y, -next_distance)
        
        # Track for visualization
        X_samples = np.vstack((X_samples, next_cmyk))
        distances.append(next_distance)
        
        # Check if we've reached the threshold
        if next_distance < threshold:
            print(f"\nReached target threshold! Distance {next_distance:.4f} < {threshold}")
            continue_bo = False
        
        # Ask user if they want to continue
        if continue_bo:
            user_input = input("\nContinue to next iteration? (y/n): ").strip().lower()
            continue_bo = user_input.startswith('y')
    
    # Show final results
    print("\n=== Final Results ===")
    best_idx = np.argmin(distances)
    best_cmyk = X_samples[best_idx]
    best_rgb = CMYK_to_RGB(best_cmyk)
    
    print(f"Total iterations: {iteration}")
    print(f"Best distance: {distances[best_idx]:.4f} (iteration {best_idx+1})")
    print(f"Best CMYK: [{', '.join([f'{v:.3f}' for v in best_cmyk])}]")
    
    # Display best volumes
    best_volumes = display_cmyk_volumes(best_cmyk, max_well_volume, "Best Volumes")
    
    # Visualize all iterations
    print("\nVisualizing all iterations...")
    X_samples_rgb = np.array([CMYK_to_RGB(cmyk) for cmyk in X_samples])
    visualize_all_iterations(X_samples_rgb, distances, target_rgb_norm)
    
    # Compare best with target
    print("Comparing best result with target...")
    compare_colors(best_cmyk, target_cmyk, ["Best Result", "Target"])
    
    return best_cmyk, target_cmyk, distances

if __name__ == "__main__":
    print("=== Interactive Bayesian Optimization Color Matcher ===")
    print("This script will simulate Bayesian Optimization for color matching")
    print("You'll see the suggested CMYK values at each iteration")
    
    run_interactive_bo()